# Shri Yogavasishtha — Chandra OCR

**Runtime:** GPU (T4 or better)  
**Steps:**
1. Run all cells in order
2. Upload the 4 PDFs when prompted (or place them in Google Drive)
3. OCR output is saved to `/content/drive/MyDrive/yogavasishtha_ocr/`

In [1]:
# ── Cell 1: Verify GPU ──────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU — switch to T4: Runtime > Change runtime type > T4 GPU')

Thu Apr 30 07:35:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# ── Cell 2: Install chandra-ocr ─────────────────────────────────────────────
!pip install -q chandra-ocr[hf]
print('chandra-ocr installed')

chandra-ocr installed


In [3]:
# ── Cell 3: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/yogavasishtha_ocr'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output will be saved to: {OUTPUT_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output will be saved to: /content/drive/MyDrive/yogavasishtha_ocr


In [4]:
# ── Cell 4: Upload PDFs ─────────────────────────────────────────────────────
from google.colab import files
import shutil

print('Select all 4 PDF files (Shri-Yogavasishtha-1.pdf through 4.pdf)')
uploaded = files.upload()

pdf_paths = []
for filename in uploaded:
    dest = f'/content/{filename}'
    if not os.path.exists(dest):
        shutil.move(filename, dest)
    pdf_paths.append(dest)
    print(f'  {filename}: {os.path.getsize(dest)/1e6:.1f} MB')

pdf_paths.sort()
print(f'\nReady: {[os.path.basename(p) for p in pdf_paths]}')

Select all 4 PDF files (Shri-Yogavasishtha-1.pdf through 4.pdf)


Saving Shri-Yogavasishtha-1.pdf to Shri-Yogavasishtha-1.pdf
Saving Shri-Yogavasishtha-2.pdf to Shri-Yogavasishtha-2.pdf
Saving Shri-Yogavasishtha-3.pdf to Shri-Yogavasishtha-3.pdf
Saving Shri-Yogavasishtha-4.pdf to Shri-Yogavasishtha-4.pdf
  Shri-Yogavasishtha-1.pdf: 1.9 MB
  Shri-Yogavasishtha-2.pdf: 1.5 MB
  Shri-Yogavasishtha-3.pdf: 1.4 MB
  Shri-Yogavasishtha-4.pdf: 3.4 MB

Ready: ['Shri-Yogavasishtha-1.pdf', 'Shri-Yogavasishtha-2.pdf', 'Shri-Yogavasishtha-3.pdf', 'Shri-Yogavasishtha-4.pdf']


In [8]:
# ── Cell 4-alt: Use Google Drive PDFs instead of uploading ──────────────────
# Run this ONLY if PDFs are already in your Drive (skip Cell 4 above)
DRIVE_PDF_DIR = '/content/drive/MyDrive/Yogavasishtha_PDFs'  # change if needed

if os.path.exists(DRIVE_PDF_DIR):
    pdf_paths = sorted([
        os.path.join(DRIVE_PDF_DIR, f)
        for f in os.listdir(DRIVE_PDF_DIR) if f.endswith('.pdf')
    ])
    for p in pdf_paths:
        print(f'Found in Drive: {p}')
else:
    print(f'Directory not found: {DRIVE_PDF_DIR}')
    print('If you already uploaded files in Cell 4, you can ignore this error.')

Directory not found: /content/drive/MyDrive/Yogavasishtha_PDFs
If you already uploaded files in Cell 4, you can ignore this error.


In [9]:
# ── Cell 5: Check page counts ───────────────────────────────────────────────
import pypdfium2 as pdfium

total_pages = 0
for pdf_path in pdf_paths:
    doc = pdfium.PdfDocument(pdf_path)
    pages = len(doc)
    doc.close()
    total_pages += pages
    print(f'{os.path.basename(pdf_path)}: {pages} pages')

est_min = total_pages / 6
print(f'\nTotal: {total_pages} pages  |  Est. ~{est_min:.0f} min on T4')

Shri-Yogavasishtha-1.pdf: 690 pages
Shri-Yogavasishtha-2.pdf: 552 pages
Shri-Yogavasishtha-3.pdf: 500 pages
Shri-Yogavasishtha-4.pdf: 866 pages

Total: 2608 pages  |  Est. ~435 min on T4


In [ ]:
# ── Cell 6: Run OCR on all 4 PDFs ───────────────────────────────────────────
import sys, time

result = subprocess.run(['which', 'chandra'], capture_output=True, text=True)
CHANDRA = result.stdout.strip()
print(f'chandra: {CHANDRA}\n')

for pdf_path in pdf_paths:
    name = os.path.splitext(os.path.basename(pdf_path))[0]
    out_dir = os.path.join(OUTPUT_DIR, name)
    os.makedirs(out_dir, exist_ok=True)

    # Skip if already has output files
    existing = [f for f in os.listdir(out_dir)
                if os.path.splitext(f)[1] in ('.md', '.txt', '.html')]
    if existing:
        print(f'SKIP {name} — {len(existing)} files already exist')
        continue

    print(f'\n━━━ Starting OCR: {name} ━━━')
    t0 = time.time()

    proc = subprocess.run([
        CHANDRA, pdf_path, out_dir,
        '--method', 'hf',
        '--no-headers-footers',
        '--max-workers', '4',
    ], text=True)

    elapsed = time.time() - t0
    status = 'DONE' if proc.returncode == 0 else f'ERROR (code {proc.returncode})'
    print(f'{status} — {elapsed/60:.1f} min')

print('\n✓ All OCR jobs complete')

chandra: /usr/local/bin/chandra


━━━ Starting OCR: Shri-Yogavasishtha-1 ━━━
DONE — 3.6 min

━━━ Starting OCR: Shri-Yogavasishtha-2 ━━━
DONE — 3.5 min

━━━ Starting OCR: Shri-Yogavasishtha-3 ━━━


In [ ]:
# ── Cell 7: Diagnose — list what chandra actually created ───────────────────
# Run this to see the actual files before merging
for pdf_path in pdf_paths:
    name = os.path.splitext(os.path.basename(pdf_path))[0]
    out_dir = os.path.join(OUTPUT_DIR, name)
    if not os.path.isdir(out_dir):
        print(f'{name}: output dir missing!')
        continue
    all_files = []
    for root, dirs, fnames in os.walk(out_dir):
        for f in fnames:
            rel = os.path.relpath(os.path.join(root, f), out_dir)
            size = os.path.getsize(os.path.join(root, f))
            all_files.append((rel, size))
    all_files.sort()
    total_kb = sum(s for _, s in all_files) / 1024
    print(f'\n{name}/  ({len(all_files)} files, {total_kb:.0f} KB total)')
    for rel, size in all_files[:10]:
        print(f'  {rel}  ({size} bytes)')
    if len(all_files) > 10:
        print(f'  ... and {len(all_files)-10} more')

In [ ]:
# ── Cell 8: Merge all pages into one .md file per part ──────────────────────
import re

TEXT_EXTS = {'.md', '.txt'}

def page_sort_key(filename):
    nums = re.findall(r'\d+', filename)
    return int(nums[-1]) if nums else 0

for pdf_path in pdf_paths:
    name = os.path.splitext(os.path.basename(pdf_path))[0]
    out_dir = os.path.join(OUTPUT_DIR, name)

    if not os.path.isdir(out_dir):
        print(f'SKIP {name} — output dir not found')
        continue

    # Collect all text files recursively
    page_files = []
    for root, dirs, fnames in os.walk(out_dir):
        for f in fnames:
            if os.path.splitext(f)[1] in TEXT_EXTS:
                page_files.append(os.path.join(root, f))

    if not page_files:
        print(f'SKIP {name} — no .md/.txt files found (run Cell 7 to check)')
        continue

    page_files.sort(key=lambda p: page_sort_key(os.path.basename(p)))

    merged_path = os.path.join(OUTPUT_DIR, f'{name}_merged.md')
    with open(merged_path, 'w', encoding='utf-8') as out_f:
        out_f.write(f'# {name}\n\n')
        for page_path in page_files:
            page_num = page_sort_key(os.path.basename(page_path))
            out_f.write(f'\n---\n<!-- page {page_num} -->\n\n')
            with open(page_path, 'r', encoding='utf-8', errors='replace') as f:
                out_f.write(f.read())

    size_kb = os.path.getsize(merged_path) / 1024
    print(f'{name}: {len(page_files)} pages merged → {merged_path} ({size_kb:.0f} KB)')

print('\n✓ Merge complete')

In [ ]:
# ── Cell 9: Preview first 2000 chars of Part 1 ──────────────────────────────
merged_part1 = os.path.join(OUTPUT_DIR, 'Shri-Yogavasishtha-1_merged.md')
if os.path.exists(merged_part1):
    with open(merged_part1, 'r', encoding='utf-8') as f:
        print(f.read(2000))
else:
    # Try to find any merged file
    merged_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('_merged.md')]
    if merged_files:
        with open(os.path.join(OUTPUT_DIR, merged_files[0]), 'r') as f:
            print(f.read(2000))
    else:
        print('No merged files found yet')

In [ ]:
# ── Cell 10 (optional): Zip merged files and download ───────────────────────
import zipfile

zip_path = '/content/yogavasishtha_ocr.zip'
merged_files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith('_merged.md')]

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in merged_files:
        zf.write(os.path.join(OUTPUT_DIR, fname), fname)

print(f'Zip: {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)')
print(f'Contains: {merged_files}')
files.download(zip_path)